# Short-Horizon Return Prediction - Memory Optimized
Lightweight pipeline for Kaggle (661k rows)

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import catboost as cb
import gc
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded!")

## 2. Load Data with Memory Optimization

In [ ]:
# Define dtype optimization
def optimize_dtypes(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != 'object':
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

# Load data
print("Loading training data...")
train_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet')
train_df = optimize_dtypes(train_df)

print("Loading test data...")
test_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet')
test_df = optimize_dtypes(test_df)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Train memory: {train_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Test memory: {test_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## 3. Data Preparation

In [ ]:
# Handle missing values
train_df.fillna(train_df.median(), inplace=True)
test_df.fillna(test_df.median(), inplace=True)

# Separate features and target
X_train = train_df.drop(['ID', 'TARGET'], axis=1)
y_train = train_df['TARGET'].values
X_test = test_df.drop(['ID'], axis=1)
test_ids = test_df['ID'].values

# Remove constant columns from BOTH train and test
const_cols_train = X_train.columns[X_train.nunique() == 1].tolist()
const_cols_test = X_test.columns[X_test.nunique() == 1].tolist()
const_cols_all = list(set(const_cols_train + const_cols_test))

if const_cols_all:
    X_train = X_train.drop(const_cols_all, axis=1, errors='ignore')
    X_test = X_test.drop(const_cols_all, axis=1, errors='ignore')
    print(f"Removed {len(const_cols_all)} constant columns")

# Ensure train and test have the same columns (handle any remaining mismatches)
common_cols = list(set(X_train.columns) & set(X_test.columns))
X_train = X_train[common_cols]
X_test = X_test[common_cols]

# Clean up memory
del train_df, test_df
gc.collect()

print(f"Training features: {X_train.shape[1]}")
print(f"Test features: {X_test.shape[1]}")
print(f"Target stats: mean={y_train.mean():.4f}, std={y_train.std():.4f}")

## 4. Quick Distribution Check

In [ ]:
# Target distribution
print(f"Target Distribution:")
print(f"  Min: {y_train.min():.4f}")
print(f"  25%: {np.percentile(y_train, 25):.4f}")
print(f"  50%: {np.percentile(y_train, 50):.4f}")
print(f"  75%: {np.percentile(y_train, 75):.4f}")
print(f"  Max: {y_train.max():.4f}")

# Plot
plt.figure(figsize=(10, 4))
plt.hist(y_train, bins=50, edgecolor='black')
plt.title('Target Distribution')
plt.xlabel('TARGET')
plt.ylabel('Frequency')
plt.show()

## 5. 3-Fold Cross-Validation (Memory-Efficient)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# Use 3 folds instead of 5 to save memory
kf = KFold(n_splits=3, shuffle=True, random_state=42)

fold_list = list(kf.split(X_train))
print(f"Using 3-fold CV (instead of 5 to save memory)")
print(f"Fold sizes: ~{len(X_train)//3:,} train, ~{len(X_train)//3:,} val per fold")

## 6. LightGBM Model

In [ ]:
# LightGBM with conservative settings to save memory
oof_lgb = np.zeros(len(X_train))
test_pred_lgb = np.zeros(len(X_test))
lgb_scores = []

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'num_leaves': 20,
    'learning_rate': 0.05,
    'max_depth': 5,
    'min_data_in_leaf': 30,
    'lambda_l1': 1.0,
    'lambda_l2': 1.0,
    'feature_fraction': 0.6,
    'bagging_fraction': 0.6,
    'random_state': 42,
    'verbose': -1,
    'num_threads': 4
}

for fold_idx, (train_idx, val_idx) in enumerate(fold_list, 1):
    print(f"\nTraining LightGBM Fold {fold_idx}/3...")
    
    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]
    y_tr = y_train[train_idx]
    y_val = y_train[val_idx]
    
    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val)
    
    model_lgb = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=500,
        valid_sets=[val_data],
        callbacks=[
            lgb.early_stopping(30),
            lgb.log_evaluation(period=0)
        ]
    )
    
    oof_lgb[val_idx] = model_lgb.predict(X_val)
    test_pred_lgb += model_lgb.predict(X_test) / 3
    
    r2_val = r2_score(y_val, oof_lgb[val_idx])
    lgb_scores.append(r2_val)
    print(f"  R² = {r2_val:.6f}")
    
    del model_lgb, X_tr, X_val, y_tr, y_val, train_data, val_data
    gc.collect()

print(f"\nLightGBM Mean R²: {np.mean(lgb_scores):.6f} (±{np.std(lgb_scores):.6f})")

## 7. CatBoost Model

In [ ]:
# CatBoost with conservative settings
oof_cb = np.zeros(len(X_train))
test_pred_cb = np.zeros(len(X_test))
cb_scores = []

cb_params = {
    'iterations': 300,
    'depth': 4,
    'learning_rate': 0.05,
    'l2_leaf_reg': 5.0,
    'random_state': 42,
    'verbose': False,
    'task_type': 'CPU',
    'thread_count': 4
}

for fold_idx, (train_idx, val_idx) in enumerate(fold_list, 1):
    print(f"\nTraining CatBoost Fold {fold_idx}/3...")
    
    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]
    y_tr = y_train[train_idx]
    y_val = y_train[val_idx]
    
    model_cb = cb.CatBoostRegressor(**cb_params)
    model_cb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=20,
        verbose=False
    )
    
    oof_cb[val_idx] = model_cb.predict(X_val)
    test_pred_cb += model_cb.predict(X_test) / 3
    
    r2_val = r2_score(y_val, oof_cb[val_idx])
    cb_scores.append(r2_val)
    print(f"  R² = {r2_val:.6f}")
    
    del model_cb, X_tr, X_val, y_tr, y_val
    gc.collect()

print(f"\nCatBoost Mean R²: {np.mean(cb_scores):.6f} (±{np.std(cb_scores):.6f})")

## 8. Linear Model (Ridge)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Ridge Regression
oof_ridge = np.zeros(len(X_train))
test_pred_ridge = np.zeros(len(X_test))
ridge_scores = []

# Fit scaler on all training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for fold_idx, (train_idx, val_idx) in enumerate(fold_list, 1):
    print(f"Training Ridge Fold {fold_idx}/3...")
    
    X_tr = X_train_scaled[train_idx]
    X_val = X_train_scaled[val_idx]
    y_tr = y_train[train_idx]
    y_val = y_train[val_idx]
    
    model_ridge = Ridge(alpha=1.0, random_state=42)
    model_ridge.fit(X_tr, y_tr)
    
    oof_ridge[val_idx] = model_ridge.predict(X_val)
    test_pred_ridge += model_ridge.predict(X_test_scaled) / 3
    
    r2_val = r2_score(y_val, oof_ridge[val_idx])
    ridge_scores.append(r2_val)
    print(f"  R² = {r2_val:.6f}")

print(f"\nRidge Mean R²: {np.mean(ridge_scores):.6f} (±{np.std(ridge_scores):.6f})")

## 9. Model Comparison & Ensembling

In [ ]:
# Summary
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)

models_summary = [
    ('LightGBM', np.mean(lgb_scores), np.std(lgb_scores)),
    ('CatBoost', np.mean(cb_scores), np.std(cb_scores)),
    ('Ridge', np.mean(ridge_scores), np.std(ridge_scores))
]

for name, mean_r2, std_r2 in models_summary:
    print(f"{name:12} | Mean R²: {mean_r2:.6f} | Std: {std_r2:.6f}")

# Compute weights
total_mean = np.mean(lgb_scores) + np.mean(cb_scores) + np.mean(ridge_scores)
lgb_weight = np.mean(lgb_scores) / total_mean
cb_weight = np.mean(cb_scores) / total_mean
ridge_weight = np.mean(ridge_scores) / total_mean

print(f"\nEnsemble Weights:")
print(f"  LightGBM: {lgb_weight*100:.1f}%")
print(f"  CatBoost: {cb_weight*100:.1f}%")
print(f"  Ridge:    {ridge_weight*100:.1f}%")

# OOF ensemble
oof_ensemble = lgb_weight * oof_lgb + cb_weight * oof_cb + ridge_weight * oof_ridge
ensemble_r2 = r2_score(y_train, oof_ensemble)

print(f"\nEnsemble OOF R²: {ensemble_r2:.6f}")
print("="*60)

## 10. Generate Test Predictions

In [ ]:
# Ensemble test predictions
test_pred_ensemble = lgb_weight * test_pred_lgb + cb_weight * test_pred_cb + ridge_weight * test_pred_ridge

print(f"\nTest Prediction Stats:")
print(f"  Mean: {test_pred_ensemble.mean():.6f}")
print(f"  Std:  {test_pred_ensemble.std():.6f}")
print(f"  Min:  {test_pred_ensemble.min():.6f}")
print(f"  Max:  {test_pred_ensemble.max():.6f}")

# Compare to training target
print(f"\nTraining Target Stats (for comparison):")
print(f"  Mean: {y_train.mean():.6f}")
print(f"  Std:  {y_train.std():.6f}")

## 11. Create and Save Submission

In [ ]:
# Create submission dataframe
submission = pd.DataFrame({
    'ID': test_ids,
    'TARGET': test_pred_ensemble
})

print(f"Submission shape: {submission.shape}")
print(f"\nFirst 10 rows:")
print(submission.head(10))

# Save to CSV
submission.to_csv('submission.csv', index=False)
print(f"\n✓ Submitted file saved as 'submission.csv'")

## 12. Final Summary

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
print(f"Dataset:")
print(f"  Training samples: {len(X_train):,}")
print(f"  Test samples: {len(X_test):,}")
print(f"  Features: {X_train.shape[1]}")
print(f"\nValidation Strategy:")
print(f"  Method: 3-Fold Cross-Validation")
print(f"  (Using 3 folds instead of 5 for memory efficiency)")
print(f"\nModel Performance (CV):")
print(f"  Best Single Model: LightGBM (R² = {np.mean(lgb_scores):.6f})")
print(f"  Ensemble R² (OOF): {ensemble_r2:.6f}")
print(f"\nExpected Private Performance:")
print(f"  ~{ensemble_r2:.4f} baseline")
print(f"  (may vary if distribution shift is significant)")
print(f"\nNext Steps:")
print(f"  1. Download 'submission.csv'")
print(f"  2. Submit to Kaggle competition")
print(f"  3. Monitor public leaderboard score")
print(f"  4. If <0.10, consider feature engineering or different model")
print(f"\n" + "="*70)